# Notebook 1 — LangChain Basics
### What is LangChain?
LangChain is a toolkit that helps you talk to AI models (like llama3.2) in a structured way.
Instead of writing raw API calls, LangChain gives you clean building blocks:
- **LLM** — the AI brain
- **Prompt Template** — a reusable message structure
- **Output Parser** — cleans up the AI's reply
- **Chain** — connects all three together with a `|` pipe

**By the end of this notebook you will know:**
1. How to call an AI model
2. How to use prompt templates
3. How to chain prompt → model → output
4. How to use memory in a conversation
5. How to build a simple multi-step chain

> Pre-requisite: Ollama installed and running with llama3.2 pulled

---
## Step 1 — Install LangChain and Ollama connector

In [1]:
# langchain        — the core library
# langchain-ollama — connects LangChain to your local Ollama model
%pip install -q langchain langchain-ollama

Note: you may need to restart the kernel to use updated packages.


---
## Step 2 — Check Ollama is Running

In [1]:
# We use Python's built-in urllib to ping the Ollama server
# Ollama runs on your machine at port 11434
import urllib.request
import json as _j

try:
    # Open a connection to Ollama's tag listing endpoint
    with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3) as r:
        # Parse the JSON response
        data = _j.loads(r.read())
        # Extract just the model names
        models = [m['name'] for m in data.get('models', [])]
        print('Ollama is running!')
        print('Models available on your machine:', models)
        if not any('llama3.2' in m for m in models):
            print('WARNING: llama3.2 not found. Run in terminal: ollama pull llama3.2')
except Exception as e:
    print('Ollama not reachable:', e)
    print('Fix: Open Ollama from system tray, or run: ollama serve')

Ollama is running!
Models available on your machine: ['llama3.2:latest']


---
## Step 3 — Your First LLM Call
This is the simplest possible thing you can do with LangChain.
Think of `ChatOllama` as the phone line to your local AI.

In [2]:
# Import ChatOllama — this is the LangChain connector to your local Ollama model
from langchain_ollama import ChatOllama

# Create the LLM object
# model='llama3.2'      — which model to use (must be pulled in Ollama)
# base_url=...          — where Ollama is running (always localhost:11434)
# temperature=0.3       — how creative the AI is (0=very focused, 1=very creative)
llm = ChatOllama(
    model='llama3.2',
    base_url='http://localhost:11434',
    temperature=0.3,
)
print('LLM created successfully')
print('Model:', llm.model)

LLM created successfully
Model: llama3.2


### Send your first message
`llm.invoke()` sends a message and waits for the reply.
It returns a message object — `.content` gives you the text.

In [3]:
# .invoke() sends a single message to the AI
# The string inside is your question or instruction
response = llm.invoke('In one sentence, what is software testing?')

# response is a message object — not just a string
# .content extracts the actual text
print('Full response object type:', type(response))
print()
print('Answer:', response.content)

Full response object type: <class 'langchain_core.messages.ai.AIMessage'>

Answer: Software testing is the process of evaluating and validating software applications to ensure they meet specific requirements, work as expected, and produce the desired results.


---
## Step 4 — Prompt Templates
A Prompt Template is a reusable message with **placeholders** (like blanks in a form).
Instead of writing the same instruction every time, you define it once and fill in the blanks.

Think of it like a test case template — structure stays the same, only the data changes.

In [4]:
# Import the prompt template class
from langchain_core.prompts import ChatPromptTemplate

# Create a template with two messages:
# 1. 'system' — gives the AI its role and behaviour rules
# 2. 'human'  — the actual user question, with {topic} as a placeholder
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a QA expert. Give short, clear answers.'),
    ('human',  'Explain {topic} in simple terms in 2 sentences.')
])

print('Prompt template created')
print()

# .invoke() on a prompt fills in the placeholders
# It returns the filled messages — not yet sent to the AI
filled_prompt = prompt.invoke({'topic': 'regression testing'})
print('Filled prompt messages:')
for msg in filled_prompt.messages:
    print(f'  [{msg.type}]: {msg.content}')

Prompt template created

Filled prompt messages:
  [system]: You are a QA expert. Give short, clear answers.
  [human]: Explain regression testing in simple terms in 2 sentences.


### Now send the filled prompt to the LLM
Notice we call `.invoke()` on the LLM with the filled prompt.

In [5]:
# Fill the placeholder with our topic
filled = prompt.invoke({'topic': 'boundary value analysis'})

# Send the filled prompt to the AI
response = llm.invoke(filled)

print('Topic: boundary value analysis')
print('Answer:', response.content)

Topic: boundary value analysis
Answer: Boundary Value Analysis (BVA) is a testing technique where you test the limits of input values to ensure that your application behaves as expected when data falls within or at the edges of its defined range. By analyzing these boundary conditions, you can identify potential issues and defects in your software before they become major problems.


---
## Step 5 — Output Parser
The AI returns a message object. An **Output Parser** extracts just the text string.
This makes it easy to use the output in the next step of your code.

In [6]:
# StrOutputParser converts the AI message object into a plain Python string
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Without parser — you get a message object
raw_response = llm.invoke('What is a test plan?')
print('Without parser, type is:', type(raw_response))
print('You have to do .content manually:', raw_response.content[:80], '...')
print()

# With parser — you get a plain string directly
parsed_response = parser.invoke(raw_response)
print('With parser, type is:', type(parsed_response))
print('Clean string output:', parsed_response[:80], '...')

Without parser, type is: <class 'langchain_core.messages.ai.AIMessage'>
You have to do .content manually: A test plan is a detailed document that outlines the approach, scope, and schedu ...

With parser, type is: <class 'langchain_core.messages.base.TextAccessor'>
Clean string output: A test plan is a detailed document that outlines the approach, scope, and schedu ...


---
## Step 6 — The Chain (the most important concept)
A **Chain** connects steps together using the `|` pipe symbol.
Each step's output becomes the next step's input — like an assembly line.

```
prompt | llm | parser
  ↓        ↓       ↓
Fill    Send    Extract
blanks  to AI   text
```
This is the core pattern you will use in every LangChain project.

In [7]:
# Build the chain by connecting steps with the pipe | operator
# Step 1: prompt  — fills in the placeholders
# Step 2: llm     — sends the filled prompt to Ollama
# Step 3: parser  — extracts the text from the response
chain = prompt | llm | parser

print('Chain created:', chain)
print()

# .invoke() on the chain runs all three steps automatically
# You just pass the placeholder values as a dictionary
result = chain.invoke({'topic': 'equivalence partitioning'})

print('Topic: equivalence partitioning')
print('Result type:', type(result))   # now it is a plain string
print('Result:', result)

Chain created: first=ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a QA expert. Give short, clear answers.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Explain {topic} in simple terms in 2 sentences.'), additional_kwargs={})]) middle=[ChatOllama(output_version=None, model='llama3.2', temperature=0.3, base_url='http://localhost:11434')] last=StrOutputParser()

Topic: equivalence partitioning
Result type: <class 'langchain_core.messages.base.TextAccessor'>
Result: Equivalence partitioning is a technique used to identify all possible input values for a piece of code that would produce the same output, allowing you to test only one value from each group. By grouping inputs into categories (or "partitions") based on 

### Try different topics with the same chain

In [8]:
# The chain is reusable — just change the input
topics = [
    'smoke testing',
    'user acceptance testing',
    'exploratory testing'
]

for topic in topics:
    # Each call to .invoke() runs the full chain: fill → send → extract
    answer = chain.invoke({'topic': topic})
    print(f'Topic: {topic}')
    print(f'Answer: {answer}')
    print('-' * 50)

Topic: smoke testing
Answer: Smoke testing is a quality assurance process where you test an application or system to ensure it's working as expected and doesn't have any major issues that would prevent it from functioning correctly. The goal of smoke testing is to verify the basic functionality, identify critical problems early on, and catch any showstoppers before they reach production.
--------------------------------------------------
Topic: user acceptance testing
Answer: User Acceptance Testing (UAT) is the process of verifying that a software or system meets the requirements and expectations of its users, ensuring it works as intended and meets their needs. It involves testing the application with real users to validate its functionality, usability, and performance, providing feedback for further improvement before release.
--------------------------------------------------
Topic: exploratory testing
Answer: Exploratory testing is an approach to testing where you manually interac

---
## Step 7 — Templates With Multiple Placeholders
You can have as many `{placeholders}` as you need in a template.

In [9]:
# Template with TWO placeholders: {domain} and {feature}
multi_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a QA analyst specialising in {domain}. '
     'Give precise, professional answers in 3 bullet points.'),
    ('human',
     'What are the key things to test for {feature}?')
])

# Build the chain
multi_chain = multi_prompt | llm | parser

# Invoke with both placeholder values
result = multi_chain.invoke({
    'domain':  'banking',      # fills {domain}
    'feature': 'fund transfer' # fills {feature}
})

print(result)

As a QA analyst specializing in banking, here are three key things to test for fund transfer:

• **Authentication and Authorization**: Verify that the user is authenticated and authorized to initiate a fund transfer before allowing the transaction to proceed. This includes testing successful login credentials, session management, and role-based access controls.

• **Transaction Amount and Limits**: Test that the system accurately processes different types of transactions (e.g., one-time, recurring, wire transfers) with varying amounts, including limits set by regulatory bodies (e.g., AML/KYC). Ensure that the system prevents or alerts on suspicious transactions exceeding these limits.

• **Fund Availability and Settlement**: Verify that the system correctly updates fund availability in real-time, taking into account factors such as transfer timing, settlement cycles, and potential delays. Test that funds are accurately credited to the recipient's account, and that the system provides c

In [10]:
# Try insurance domain
result2 = multi_chain.invoke({
    'domain':  'insurance',
    'feature': 'claims submission'
})
print(result2)

As a QA analyst specializing in insurance, here are three key things to test for claims submission:

• **Policy Coverage and Eligibility**: Verify that the claimant's policy covers the specific loss or damage they're claiming. Test that the claimant meets the eligibility criteria, such as deductibles, exclusions, and coverage limits. Ensure that the system correctly identifies any potential issues with the policy.

• **Claim Data Accuracy and Completeness**: Validate that the claimant provides accurate and complete information, including dates, times, locations, and descriptions of the incident or loss. Test that the system checks for missing or incomplete fields, and that the claim is not processed without all required information.

• **System Processing and Payment Rules**: Simulate different scenarios to test how the system processes claims according to payment rules, such as coinsurance, depreciation, or other applicable provisions. Verify that the system applies the correct calcul

---
## Step 8 — Conversation Memory
By default the AI remembers nothing between calls.
To have a real conversation, you must pass the history yourself.
LangChain uses a list of messages to represent conversation history.

In [11]:
# HumanMessage  — a message from the user
# AIMessage     — a message from the AI
# SystemMessage — sets the AI's role at the start
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Start with a system message setting the context
conversation_history = [
    SystemMessage(content='You are a helpful QA assistant. Keep answers short.')
]

print('Conversation history initialised')
print('Messages so far:', len(conversation_history))

Conversation history initialised
Messages so far: 1


In [12]:
def chat(user_input: str) -> str:
    """
    Send a message and remember the conversation.
    Every call adds the user message and AI reply to history.
    """
    # Add the user's message to history
    conversation_history.append(HumanMessage(content=user_input))

    # Send the ENTIRE history to the AI (so it remembers context)
    response = llm.invoke(conversation_history)

    # Add the AI's reply to history for next turn
    conversation_history.append(AIMessage(content=response.content))

    return response.content

print('chat() function ready')

chat() function ready


In [13]:
# Turn 1 — ask a question
reply1 = chat('What is regression testing?')
print('You: What is regression testing?')
print('AI :', reply1)
print()

You: What is regression testing?
AI : Regression testing is the process of re-testing software after changes have been made to ensure that no new bugs were introduced and existing functionality still works as expected.



In [14]:
# Turn 2 — AI remembers we were talking about regression testing
reply2 = chat('When should I run it?')
print('You: When should I run it?')
print('AI :', reply2)
print()

# Show the full history
print(f'Total messages in history: {len(conversation_history)}')
for msg in conversation_history:
    label = type(msg).__name__
    print(f'  [{label}]: {msg.content[:80]}')

You: When should I run it?
AI : Run regression testing:

1. After major code changes or updates
2. Before releasing a new version or patch
3. Periodically (e.g., weekly, bi-weekly) to catch issues early on.

Total messages in history: 5
  [SystemMessage]: You are a helpful QA assistant. Keep answers short.
  [HumanMessage]: What is regression testing?
  [AIMessage]: Regression testing is the process of re-testing software after changes have been
  [HumanMessage]: When should I run it?
  [AIMessage]: Run regression testing:

1. After major code changes or updates
2. Before releas


---
## Step 9 — Sequential Chain (Output of one feeds the next)
You can chain two separate chains together.
The output of Chain 1 becomes the input of Chain 2.
This is the foundation of multi-agent pipelines.

In [15]:
# Chain 1 — Generate a test scenario description
chain1_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a QA analyst. Write a one-paragraph test scenario.'),
    ('human',  'Write a test scenario for: {feature}')
])
chain1 = chain1_prompt | llm | parser

# Chain 2 — Take that scenario and generate test steps
chain2_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a QA engineer. Convert test scenarios into numbered test steps.'),
    ('human',  'Convert this scenario into step-by-step test steps:\n\n{scenario}')
])
chain2 = chain2_prompt | llm | parser

# Run Chain 1
feature = 'login with two-factor authentication for a banking app'
print('Step 1: Generating scenario...')
scenario = chain1.invoke({'feature': feature})
print('Scenario generated:')
print(scenario)
print()

Step 1: Generating scenario...
Scenario generated:
Here's a test scenario for logging in to a banking app with two-factor authentication:

**Test Scenario:** "Login with Two-Factor Authentication"

**Preconditions:** The user has created an account in the banking app and has enabled two-factor authentication (2FA) on their account. The 2FA method is set to SMS-based, where a verification code is sent to the user's registered mobile number.

**Test Steps:**

1. Launch the banking app and navigate to the login screen.
2. Enter the correct username and password for the account.
3. Click the "Login" button.
4. The app should prompt the user to enter a verification code sent via SMS.
5. Enter the received verification code in the designated field.
6. If the entered code is correct, the app should authenticate the user and display the dashboard.
7. Verify that the 2FA feature is successfully enabled and that the user's account is secured with an additional layer of authentication.

**Expecte

In [16]:
# Run Chain 2 using Chain 1's output as input
print('Step 2: Converting scenario to test steps...')
test_steps = chain2.invoke({'scenario': scenario})
print('Test Steps:')
print(test_steps)

Step 2: Converting scenario to test steps...
Test Steps:
Here are the test steps in a numbered format:

**Test Scenario:** "Login with Two-Factor Authentication"

**Preconditions:** The user has created an account in the banking app and has enabled two-factor authentication (2FA) on their account. The 2FA method is set to SMS-based, where a verification code is sent to the user's registered mobile number.

**Test Steps:**

1. **Launch Banking App**: Launch the banking app on a device with a stable internet connection.
2. **Navigate to Login Screen**: Navigate to the login screen within the app.
3. **Enter Correct Credentials**: Enter the correct username and password for the account in the designated fields.
4. **Click Login Button**: Click the "Login" button to proceed with the login process.
5. **Verify 2FA Prompt**: Verify that a verification code prompt appears on the screen, indicating that two-factor authentication is required.
6. **Enter Verification Code**: Enter the received v

---
## Summary — What You Learned

| Concept | What it is | Code |
|---|---|---|
| LLM | The AI brain | `ChatOllama(model='llama3.2')` |
| Prompt Template | Reusable message with placeholders | `ChatPromptTemplate.from_messages([...])` |
| Output Parser | Converts AI response to plain string | `StrOutputParser()` |
| Chain | Connects steps with pipe | `prompt | llm | parser` |
| Memory | Pass history list to remember context | `llm.invoke(conversation_history)` |
| Sequential | Output of chain feeds next chain | `chain1 output → chain2 input` |

**You are now ready for Notebook 2 — LangGraph Basics.**